## **Farmland Birds Under Pressure**
# Biodiversity Trends and Agricultural Change in Europe

**Main question**

**How have common farmland-bird populations changed across European countries since 2000, and are agricultural intensity, organic farming and protected-area coverage associated with different national trajectories?**

**Layer 1:** Long-term biodiversity trends

Use:

- env_bio2: national farmland-bird indices
- env_bio3: EU bird-group indices and uncertainty

Source: https://ec.europa.eu/eurostat/web/main/data/database#Updates


**Period:**

1990–2019 for Germany and long-term countries
2000–2019 for the main cross-country comparison
Later years only in a clearly labelled supplementary analysis

**Questions:**

- Which countries experienced the largest decline since 2000?
- Did Germany decline faster than the EU benchmark?
- Are farmland birds declining faster than forest birds?
- Are there countries showing stabilization or recovery?
- How sensitive are rankings to the chosen endpoint?

 **Bird groups**
CO_ALL: all common bird species
CO_FARM: common farmland bird species
CO_FOR: common forest bird species

**Estimate types**
NSME: unsmoothed estimate
SME: smoothed estimate
SME_LW95: lower 95% confidence limit
SME_UP95: upper 95% confidence limit

 **Index reference systems**
I00: 2000 = 100
I90: 1990 = 100
I_LY: latest year = 100


**Layer 2: Environmental drivers**
Add three Eurostat indicators.

1. Organic farming

Dataset:[Area under organic farming — sdg_02_40](https://ec.europa.eu/eurostat/databrowser/view/tag00025/default/table?lang=en)

## 1. Load the Eurostat bulk-download files

Eurostat's TSV files are compressed with gzip. Pandas can read them directly; manual decompression is unnecessary. The path resolver below works both next to the notebook and in the original Google Drive folder.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


def find_data_file(filename):
    # Return the first existing copy of a source file.
    candidates = [
        Path(filename),
        Path("upload") / filename,
        Path("/content/drive/MyDrive/Final_Project") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


source_paths = {
    "env_bio2": find_data_file("estat_env_bio2.tsv.gz"),
    "env_bio3": find_data_file("estat_env_bio3.tsv.gz"),
    "env_bio4": find_data_file("estat_env_bio4.tsv.gz"),
}

raw_bio2 = pd.read_csv(source_paths["env_bio2"], sep="	", compression="gzip", dtype=str)
raw_bio3 = pd.read_csv(source_paths["env_bio3"], sep="	", compression="gzip", dtype=str)
raw_bio4 = pd.read_csv(source_paths["env_bio4"], sep="	", compression="gzip", dtype=str)

pd.DataFrame(
    {
        "dataset": source_paths.keys(),
        "rows": [len(raw_bio2), len(raw_bio3), len(raw_bio4)],
        "columns": [raw_bio2.shape[1], raw_bio3.shape[1], raw_bio4.shape[1]],
        "source_file": [str(path) for path in source_paths.values()],
    }
)


## 2. Parse Eurostat dimensions, values, and flags

The first column is not one variable. It contains several comma-separated dimensions, while `\TIME_PERIOD` marks the boundary between those dimensions and the year columns.

The raw missing-value check can be misleading: Eurostat represents missing observations with `:` rather than a conventional null. Cells can also contain a number followed by a status flag, for example `60.84 d`. The function below:

1. separates the first column into named dimensions;
2. reshapes years from columns into rows;
3. extracts the numeric observation;
4. keeps Eurostat's status flag in a separate column; and
5. converts `:` to a true missing value.


In [ ]:
def eurostat_to_long(raw_df, dimension_names, dataset_name):
    # Convert a Eurostat bulk TSV table to tidy long format.
    id_column = raw_df.columns[0]
    expected_dimensions = len(dimension_names)

    dimensions = raw_df[id_column].str.split(",", expand=True)
    if dimensions.shape[1] != expected_dimensions:
        raise ValueError(
            f"{dataset_name}: expected {expected_dimensions} dimensions "
            f"but found {dimensions.shape[1]}"
        )
    dimensions.columns = dimension_names

    separated = pd.concat(
        [dimensions, raw_df.drop(columns=id_column)],
        axis=1,
    )

    tidy = separated.melt(
        id_vars=dimension_names,
        var_name="year",
        value_name="raw_value",
    )

    tidy["year"] = pd.to_numeric(tidy["year"].str.strip(), errors="raise").astype("int64")
    tidy["raw_value"] = tidy["raw_value"].astype("string").str.strip()

    extracted = tidy["raw_value"].str.extract(
        r"^(?:([-+]?\d+(?:\.\d+)?)|:)\s*([A-Za-z]+)?$"
    )
    tidy["value"] = pd.to_numeric(extracted[0], errors="coerce")
    tidy["flag"] = extracted[1].astype("string")
    tidy["dataset"] = dataset_name

    unexpected = tidy.loc[
        ~tidy["raw_value"].str.match(
            r"^(?:([-+]?\d+(?:\.\d+)?)|:)\s*([A-Za-z]+)?$",
            na=False,
        ),
        "raw_value",
    ].drop_duplicates()
    if not unexpected.empty:
        raise ValueError(f"{dataset_name}: unparsed cells: {unexpected.tolist()}")

    return tidy[
        ["dataset", *dimension_names, "year", "value", "flag", "raw_value"]
    ].sort_values([*dimension_names, "year"], ignore_index=True)


bio2_long = eurostat_to_long(
    raw_bio2,
    ["frequency", "unit", "country"],
    "env_bio2",
)

bio3_long = eurostat_to_long(
    raw_bio3,
    ["frequency", "estimate_type", "bird_group", "unit", "country"],
    "env_bio3",
)

bio4_long = eurostat_to_long(
    raw_bio4,
    ["frequency", "unit", "protected_area_type", "country"],
    "env_bio4",
)

bio2_long.head()


## 3. Validate the tidy datasets

Duplicates should be checked using the dimensions that uniquely identify an observation, not across the original wide rows. Missing observations should be counted after `:` has been converted to `NaN`.


In [ ]:
def dataset_audit(df, key_columns):
    return {
        "rows": len(df),
        "valid_values": int(df["value"].notna().sum()),
        "missing_values": int(df["value"].isna().sum()),
        "flagged_values": int(df["flag"].notna().sum()),
        "duplicate_keys": int(df.duplicated(key_columns).sum()),
        "first_year": int(df["year"].min()),
        "last_year": int(df["year"].max()),
    }


audit = pd.DataFrame.from_dict(
    {
        "env_bio2": dataset_audit(
            bio2_long, ["frequency", "unit", "country", "year"]
        ),
        "env_bio3": dataset_audit(
            bio3_long,
            ["frequency", "estimate_type", "bird_group", "unit", "country", "year"],
        ),
        "env_bio4": dataset_audit(
            bio4_long,
            ["frequency", "unit", "protected_area_type", "country", "year"],
        ),
    },
    orient="index",
).rename_axis("dataset").reset_index()

audit


## 4. Select the variables needed for the study

### National farmland-bird series (`env_bio2`)

This dataset is already restricted to the annual index with 2000 as its reference. Missing observations are removed from the analytical subset, but flagged observations are retained so that sensitivity checks remain possible.

### EU benchmark (`env_bio3`)

We select common farmland species and the 2000-based index. The unsmoothed estimate, smoothed estimate, and its lower and upper 95% confidence limits are pivoted into separate columns. This produces one EU row per year.

### Protected areas (`env_bio4`)

We use the percentage of terrestrial protected area. Marine protected areas and square-kilometre totals are not appropriate predictors for a national farmland-bird model.


In [ ]:
birds_country = (
    bio2_long.loc[
        bio2_long["frequency"].eq("A")
        & bio2_long["unit"].eq("I00")
        & bio2_long["value"].notna(),
        ["country", "year", "value", "flag"],
    ]
    .rename(columns={"value": "bird_index", "flag": "bird_flag"})
    .sort_values(["country", "year"], ignore_index=True)
)

estimate_names = {
    "NSME": "eu_unsmoothed",
    "SME": "eu_smoothed",
    "SME_LW95": "eu_lower_95",
    "SME_UP95": "eu_upper_95",
}

eu_bird_benchmark = (
    bio3_long.loc[
        bio3_long["frequency"].eq("A")
        & bio3_long["country"].eq("EU27_2020")
        & bio3_long["unit"].eq("I00")
        & bio3_long["bird_group"].eq("CO_FARM")
        & bio3_long["estimate_type"].isin(estimate_names)
        & bio3_long["value"].notna(),
        ["year", "estimate_type", "value"],
    ]
    .pivot(index="year", columns="estimate_type", values="value")
    .rename(columns=estimate_names)
    .rename_axis(columns=None)
    .reset_index()
    .sort_values("year", ignore_index=True)
)

protected_terrestrial = (
    bio4_long.loc[
        bio4_long["frequency"].eq("A")
        & bio4_long["unit"].eq("PC")
        & bio4_long["protected_area_type"].eq("TPA")
        & bio4_long["country"].ne("EU27_2020")
        & bio4_long["value"].notna(),
        ["country", "year", "value", "flag"],
    ]
    .rename(
        columns={
            "value": "protected_terrestrial_pct",
            "flag": "protected_area_flag",
        }
    )
    .sort_values(["country", "year"], ignore_index=True)
)

selection_summary = pd.DataFrame(
    {
        "dataframe": ["birds_country", "eu_bird_benchmark", "protected_terrestrial"],
        "rows": [len(birds_country), len(eu_bird_benchmark), len(protected_terrestrial)],
        "first_year": [
            birds_country["year"].min(),
            eu_bird_benchmark["year"].min(),
            protected_terrestrial["year"].min(),
        ],
        "last_year": [
            birds_country["year"].max(),
            eu_bird_benchmark["year"].max(),
            protected_terrestrial["year"].max(),
        ],
    }
)

selection_summary


## 5. Merge compatible information

The EU benchmark is joined by year. Protected-area coverage is joined by country and year. A left join preserves every valid national bird observation and makes unavailable predictors visible instead of silently deleting those rows.

The resulting `bird_panel` is useful for descriptive comparisons. `model_panel_observed` is the complete-case subset for models that require protected-area coverage. No values are interpolated or filled at this stage.


In [ ]:
bird_panel = (
    birds_country
    .merge(
        eu_bird_benchmark,
        on="year",
        how="left",
        validate="many_to_one",
    )
    .merge(
        protected_terrestrial,
        on=["country", "year"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(["country", "year"], ignore_index=True)
)

bird_panel["bird_minus_eu"] = (
    bird_panel["bird_index"] - bird_panel["eu_smoothed"]
)
bird_panel["bird_change_pct"] = (
    bird_panel.groupby("country")["bird_index"].pct_change(fill_method=None) * 100
)

model_panel_observed = (
    bird_panel.dropna(
        subset=["bird_index", "eu_smoothed", "protected_terrestrial_pct"]
    )
    .reset_index(drop=True)
)

merge_audit = pd.Series(
    {
        "national bird rows": len(bird_panel),
        "rows with EU benchmark": int(bird_panel["eu_smoothed"].notna().sum()),
        "rows with protected-area value": int(
            bird_panel["protected_terrestrial_pct"].notna().sum()
        ),
        "complete rows for current model": len(model_panel_observed),
        "countries in current model": model_panel_observed["country"].nunique(),
    },
    name="count",
).to_frame()

merge_audit


## 6. Create country-level trend features

The primary comparison window is 2000–2019 because it preserves a common reference year and includes Germany's latest available bird observation. For each country with sufficient observations, the table below calculates:

- the first and last available index in the window;
- total percentage change;
- an ordinary least-squares slope in index points per year;
- coverage and flag counts; and
- the country's 2019 position relative to the EU smoothed index.

The slope summarizes direction; it should not be interpreted as a population forecast.


In [ ]:
def summarize_country_trend(group, start_year=2000, end_year=2019, min_years=10):
    sample = group.loc[
        group["year"].between(start_year, end_year),
        ["year", "bird_index", "bird_flag"],
    ].dropna(subset=["bird_index"]).sort_values("year")

    if len(sample) < min_years:
        return pd.Series(
            {
                "first_year": pd.NA,
                "last_year": pd.NA,
                "first_index": np.nan,
                "last_index": np.nan,
                "change_pct": np.nan,
                "slope_points_per_year": np.nan,
                "n_years": len(sample),
                "flagged_years": int(sample["bird_flag"].notna().sum()),
            }
        )

    first = sample.iloc[0]
    last = sample.iloc[-1]
    slope = np.polyfit(sample["year"], sample["bird_index"], 1)[0]

    return pd.Series(
        {
            "first_year": int(first["year"]),
            "last_year": int(last["year"]),
            "first_index": first["bird_index"],
            "last_index": last["bird_index"],
            "change_pct": (last["bird_index"] / first["bird_index"] - 1) * 100,
            "slope_points_per_year": slope,
            "n_years": len(sample),
            "flagged_years": int(sample["bird_flag"].notna().sum()),
        }
    )


country_trends_2000_2019 = (
    birds_country.groupby("country")[["year", "bird_index", "bird_flag"]]
    .apply(summarize_country_trend)
    .reset_index()
    .dropna(subset=["change_pct"])
    .sort_values("change_pct", ignore_index=True)
)

country_trends_2000_2019


## 7. Initial visual checks

The first plot compares Germany with the EU smoothed farmland-bird index and its 95% confidence interval. The second plot ranks national changes over the common 2000–2019 study period.


In [ ]:
germany = birds_country.query("country == 'DE'")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(germany["year"], germany["bird_index"], label="Germany", linewidth=2.3)
ax.plot(
    eu_bird_benchmark["year"],
    eu_bird_benchmark["eu_smoothed"],
    label="EU-27 smoothed estimate",
    linewidth=2.3,
)
ax.fill_between(
    eu_bird_benchmark["year"],
    eu_bird_benchmark["eu_lower_95"],
    eu_bird_benchmark["eu_upper_95"],
    alpha=0.18,
    label="EU-27 95% confidence interval",
)
ax.axhline(100, color="grey", linestyle="--", linewidth=1, label="2000 reference")
ax.set(title="Common farmland bird index", xlabel="Year", ylabel="Index (2000 = 100)")
ax.legend(frameon=False, ncols=2)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
plot_data = country_trends_2000_2019.sort_values("change_pct")

fig, ax = plt.subplots(figsize=(10, 7))
colors = np.where(plot_data["change_pct"] < 0, "#9F3A38", "#2A7F62")
ax.barh(plot_data["country"], plot_data["change_pct"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(
    title="Change in national common farmland bird indices",
    xlabel="Change from first to last available observation, 2000–2019 (%)",
    ylabel="Eurostat country code",
)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 8. Export analysis-ready DataFrames

The exports keep the data at clearly defined grains:

- `birds_country_long.csv`: one national bird index per country-year;
- `eu_farmland_bird_benchmark.csv`: one EU benchmark row per year;
- `protected_terrestrial_long.csv`: one protected-area percentage per country-year;
- `bird_country_year_panel.csv`: merged descriptive panel, including missing predictors;
- `bird_model_panel_observed.csv`: complete rows for the current protected-area model;
- `country_trends_2000_2019.csv`: country-level trend summary; and
- `data_quality_audit.csv`: parsing and quality-control counts.


In [ ]:
output_dir = Path("outputs/dataframes")
output_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "birds_country_long.csv": birds_country,
    "eu_farmland_bird_benchmark.csv": eu_bird_benchmark,
    "protected_terrestrial_long.csv": protected_terrestrial,
    "bird_country_year_panel.csv": bird_panel,
    "bird_model_panel_observed.csv": model_panel_observed,
    "country_trends_2000_2019.csv": country_trends_2000_2019,
    "data_quality_audit.csv": audit,
}

for filename, dataframe in exports.items():
    dataframe.to_csv(output_dir / filename, index=False)

export_summary = pd.DataFrame(
    {
        "file": exports.keys(),
        "rows": [len(df) for df in exports.values()],
        "columns": [df.shape[1] for df in exports.values()],
    }
)

export_summary


## 9. Recommended next stage

The present panel can describe national bird trajectories and their relationship with protected-area coverage. It is not yet a robust agricultural-pressure model because protected-area percentage changes slowly and is only available from 2011.

The next additions should be reshaped with the same function and merged on `country` and `year`:

1. `sdg_02_40`: percentage of utilised agricultural area under organic farming;
2. `aei_pr_gnb`: nitrogen balance per hectare of utilised agricultural area; and
3. optionally `aei_fm_salpest09`: pesticide sales, normalized by utilised agricultural area.

Before fitting a model:

- measure country-year overlap after every join;
- keep flags rather than discarding them silently;
- avoid treating an index as an absolute bird count;
- compare a country-and-year baseline with models containing environmental indicators;
- use a time-based or country-grouped validation split rather than a random row split; and
- describe model results as associations, not causal effects.
